In [9]:
df = spark.read.format("delta").load(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/energy_base_dataset/"
)

In [10]:
df.show(5)
df.printSchema()

In [11]:
from pyspark.sql.functions import *
df = df.withColumn("hour", hour("tstp")) \
       .withColumn("day_of_week", dayofweek("tstp")) \
       .withColumn("month", month("tstp")) \
       .withColumn("quarter", quarter("tstp")) \
       .withColumn("week_of_year", weekofyear("tstp")) \
       .withColumn("day_of_year", dayofyear("tstp"))

df = df.withColumn(
"season",
when(col("month").isin(12,1,2),"winter")
.when(col("month").isin(3,4,5),"spring")
.when(col("month").isin(6,7,8),"summer")
.otherwise("autumn")
)

df = df.withColumn(
"peak_hour",
when((col("hour")>=7)&(col("hour")<=10),1)
.when((col("hour")>=17)&(col("hour")<=21),1)
.otherwise(0)
)

In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead

window = Window.partitionBy("LCLid").orderBy("tstp")

df = df.withColumn("lag_1h", lag("energy_kwh", 2).over(window)) \
       .withColumn("lag_24h", lag("energy_kwh", 48).over(window)) \
       .withColumn("lag_7d", lag("energy_kwh", 336).over(window))

df = df.withColumn(
"target_energy_next_30min",
lead("energy_kwh", 1).over(window)
)

df = df.dropna(subset=[
"lag_1h",
"lag_24h",
"lag_7d",
"target_energy_next_30min"
])




In [13]:
df.write.format("delta") \
.mode("overwrite") \
.save(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/temporal_features/"
)

In [14]:
# test pour voir si ma partie est bonne 
dff = spark.read.format("delta").load(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/temporal_features_test/"
)
dff = dff.limit(100000)
dff.show(5)
dff.printSchema()